In [2]:
import pandas as pd
import folium
from geopy.geocoders import Nominatim # <--- Change this import
from geopy.extra.rate_limiter import RateLimiter # <--- Add this import
import csv
import json
from folium.plugins import MarkerCluster

# 加载国家边界数据（假设你有一个 countries.geojson 文件）
with open('custom.geo.json', 'r', encoding='utf-8') as f:
    countries_geojson = json.load(f)

# 定义颜色列表
colors = ['#742368','#FCC223','#19978C','#F7A95E','#65C1C0']

# 从外部文件加载指定国家的颜色
with open('specified_colors.json', 'r', encoding='utf-8') as f:
    specified_colors = json.load(f)

# 构建邻接表
adjacency_dict = {}
for feature in countries_geojson['features']:
    country_name = feature['properties']['name']
    neighbors = set()
    for neighbor_feature in countries_geojson['features']:
        # 判断两个国家是否相邻
        if any(border in neighbor_feature['geometry']['coordinates'][0] 
               for border in feature['geometry']['coordinates'][0]):
            neighbors.add(neighbor_feature['properties']['name'])
    adjacency_dict[country_name] = neighbors

# 读取外部 cities_list 文件
with open('cities_list.json', 'r', encoding='utf-8') as f:
    cities_list = json.load(f)

# 自动生成 visited_countries 集合
visited_countries = {country for _, country in cities_list}

# 四色定理算法函数
def four_color_algorithm(adjacency_dict, specified_colors):
    country_colors = specified_colors.copy()  # 先复制指定的颜色
    remaining_colors = [color for color in colors if color not in specified_colors.values()]

    def find_available_color(country, colors, adjacency_dict, country_colors):
        used_colors = set()
        for neighbor in adjacency_dict.get(country, set()):
            if neighbor in country_colors:
                used_colors.add(country_colors[neighbor])
        available_colors = [color for color in colors if color not in used_colors]
        return available_colors[0] if available_colors else None

    # 对未指定颜色的国家进行着色
    for country in adjacency_dict.keys():
        if country not in country_colors:
            available_color = find_available_color(country, remaining_colors, adjacency_dict, country_colors)
            if available_color is not None:
                country_colors[country] = available_color
            else:
                country_colors[country] = 'gray'  # 如果没有可用的颜色，则默认灰色

    return country_colors

# 应用四色定理算法为所有国家分配颜色
country_colors = four_color_algorithm(adjacency_dict, specified_colors)

# 将未访问过的国家颜色设置为灰色
for country in adjacency_dict.keys():
    if country not in visited_countries:
        country_colors[country] = 'gray'

def geocode_cities(cities):
    geolocator = Nominatim(user_agent="my-travel-map-app", timeout=10)
    geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)
    locations = []
    
    for city, country in cities:
        query = f"{city}, {country}"
        try:
            location = geocode(query, exactly_one=True, addressdetails=True, language='en')
            if location is not None:
                # Use .get() to safely access nested dictionary keys
                city_name = location.raw.get('address', {}).get('city', city)
                country_name = location.raw.get('address', {}).get('country', country)
                latitude = location.latitude
                longitude = location.longitude
                locations.append((city_name, country_name, latitude, longitude))
            else:
                print(f"Could not find coordinates for: {query}")
        except Exception as e:
            print(f"Error geocoding {query}: {e}")
    
    return locations


coordinates = geocode_cities(cities_list)

# Output the data to a CSV file
output_file = "city_coordinates.csv"

with open(output_file, mode="w", newline="", encoding="utf-8") as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(["City", "Country", "Longitude", "Latitude"])  # Write header
    for city, country, lat, lon in coordinates:
        writer.writerow([city, country, lon, lat])

# Read data from the CSV file
data = pd.read_csv("city_coordinates.csv")

# Create a Folium Map
world_map = folium.Map(location=[0, 0], zoom_start=2, tiles="OpenStreetMap")

# 为每个国家着色
folium.GeoJson(
    countries_geojson,
    style_function=lambda feature: {
        'fillColor': country_colors.get(feature['properties']['name'], 'gray'),  # 默认灰色
        'color': 'black',
        'weight': 1,
        'fillOpacity': 0.5
    }
).add_to(world_map)

# 创建聚类标记
marker_cluster = MarkerCluster().add_to(world_map)

# 添加城市标记
for index, row in data.iterrows():
    city = row["City"]
    country = row["Country"]
    lat = row["Latitude"]
    lon = row["Longitude"]
    tooltip_text = f"{city}, {country}"
    folium.Marker(
        location=[lat, lon],
        tooltip=tooltip_text,
        icon=folium.Icon(color='blue')
    ).add_to(marker_cluster)

# Save the map to an HTML file
world_map.save("liteng_footprint.html")